## Load the Cleaned Dataset

In [1]:
import pandas as pd
df = pd.read_csv(
    "../data/processed/cleaned_demand_timeseries.csv",
    parse_dates = [0],
    index_col = 0
)

df.head()

C:\Users\ariel\AppData\Local\Temp\ipykernel_35896\3539524339.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(
C:\Users\ariel\AppData\Local\Temp\ipykernel_35896\3539524339.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(


,demand,demand_diff
1,20167,249.0
2,20328,161.0
3,19460,-868.0
4,18654,-806.0
5,18248,-406.0


## Creat Lag features

In [2]:
# Previous half hour
df["lag_1"] = df["demand"].shift(1)

# Same time previous day
df["lag_48"] = df["demand"].shift(48)

df[["demand", "lag_1", "lag_48"]].head(60)

,demand,lag_1,lag_48
1,20167,NaN,NaN
2,20328,20167.0,NaN
3,19460,20328.0,NaN
4,18654,19460.0,NaN
5,18248,18654.0,NaN
6,17855,18248.0,NaN
7,17367,17855.0,NaN
8,16774,17367.0,NaN
9,16489,16774.0,NaN
10,16289,16489.0,NaN


## Create Rolling Statistics

In [3]:
# Rolling mean (daily window)
df["rolling_mean_48"] = df["demand"].rolling(window=48).mean()

df[["demand", "rolling_mean_48"]].head(100)


,demand,rolling_mean_48
1,20167,NaN
2,20328,NaN
3,19460,NaN
4,18654,NaN
5,18248,NaN
...,...,...
96,25564,29080.666667
97,25674,29153.604167
98,25355,29227.541667
99,24800,29302.958333


## Extract Date & Time Components

In [4]:
# Critical features for XGBoost and LSTM
dt_idx = pd.DatetimeIndex(df.index)

df["hour"] = dt_idx.hour
df["day_of_week"] = dt_idx.dayofweek
df["month"] = dt_idx.month

df[["hour", "day_of_week", "month"]].head()


,hour,day_of_week,month
1,0,3,1
2,0,3,1
3,0,3,1
4,0,3,1
5,0,3,1


## Drop rows with NaNs (Created by Lags)

In [5]:
df_fe = df.dropna()

df_fe.isna().sum()

demand             0
demand_diff        0
lag_1              0
lag_48             0
rolling_mean_48    0
hour               0
day_of_week        0
month              0
dtype: int64

## Scale the clean feature matrix

In [8]:
from sklearn.preprocessing import MinMaxScaler

feature_cols = [
    "demand",
    "lag_1",
    "lag_48",
    "rolling_mean_48",
    "hour",
    "day_of_week",
    "month"
]

# Time-aware split (80/20)
split_index = int(len(df_fe) * 0.8)
train_fe = df_fe.iloc[:split_index]
test_fe = df_fe.iloc[split_index:]

# Fit scaler ONLY on training data
scaler = MinMaxScaler()
train_scaled = train_fe.copy()
test_scaled = test_fe.copy()

train_scaled[feature_cols] = scaler.fit_transform(train_fe[feature_cols])
test_scaled[feature_cols] = scaler.transform(test_fe[feature_cols])


## Save Feature-Engineered Dataset

In [9]:
# Save unscaled features (for tree models / ARIMA-style ML)
df_fe.to_csv("../data/processed/feature_engineered_unscaled.csv")

# Save scaled features (for LSTM / neural networks)
train_scaled.to_csv("../data/processed/feature_engineered_train_scaled.csv")
test_scaled.to_csv("../data/processed/feature_engineered_test_scaled.csv")

df_fe.head()

,demand,demand_diff,lag_1,lag_48,rolling_mean_48,hour,day_of_week,month
49,22173,724.0,21449.0,20167.0,23569.395833,0,3,1
50,21806,-367.0,22173.0,20328.0,23600.187500,0,3,1
51,21180,-626.0,21806.0,19460.0,23636.020833,0,3,1
52,20877,-303.0,21180.0,18654.0,23682.333333,0,3,1
53,20476,-401.0,20877.0,18248.0,23728.750000,0,3,1
